# Flat price model
Train once, then reuse the saved pipeline for inference.


In [ ]:
# 1. Imports & configuration
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

DATA_PATH = Path("flats.csv")
MODEL_PATH = Path("xgb_flat_price_model.joblib")
RANDOM_STATE = 42
FORCE_RETRAIN = False  # set True to retrain even if the saved model exists


In [ ]:
# 2. Data loading and cleaning
def load_and_clean(data_path: Path = DATA_PATH):
    df = pd.read_csv(data_path)

    df = df.rename(columns={
        "Etaj": "floor",
        "Xonalar soni": "rooms",
        "Qurilish turi": "building_type",
        "Maydon(maydon eng kami sifatida)": "area",
        "Arzon": "price_low",
        "Bozor": "price_market",
        "Qimmat": "price_high"
    })

    df = df.dropna(subset=["price_market", "floor", "rooms", "building_type", "area"])
    df = df[df["area"] > 5]
    df = df[df["price_market"] > 0]

    feature_cols = ["floor", "rooms", "building_type", "area"]
    target_col = "price_market"

    return df[feature_cols], df[target_col]


In [ ]:
# 3. Pipeline factory
def build_pipeline():
    numeric_features = ["floor", "rooms", "area"]
    categorical_features = ["building_type"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", xgb_model),
    ])


In [ ]:
# 4. Train (once) or load existing model
if MODEL_PATH.exists() and not FORCE_RETRAIN:
    model = joblib.load(MODEL_PATH)
    training_metrics = None
    print(f"Loaded existing model from {MODEL_PATH}")
else:
    X, y = load_and_clean(DATA_PATH)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE
    )

    model = build_pipeline()
    model.fit(X_train, y_train)

    def _scores(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        return {"MAE": mae, "RMSE": rmse}

    training_metrics = {
        "validation": _scores(y_valid, model.predict(X_valid)),
        "test": _scores(y_test, model.predict(X_test)),
    }

    for split, scores in training_metrics.items():
        print(f"{split.title()} - MAE: {scores['MAE']:.2f} | RMSE: {scores['RMSE']:.2f}")

    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, MODEL_PATH)
    print(f"Saved model to {MODEL_PATH}")


In [ ]:
# 5. Reusable prediction helper
REQUIRED_FEATURES = ["floor", "rooms", "building_type", "area"]

def predict_price(records, trained_model=None):
    mdl = trained_model or model
    df = pd.DataFrame(records)
    missing = set(REQUIRED_FEATURES) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required feature columns: {sorted(missing)}")
    return mdl.predict(df)

# Example inference
example_input = [{
    "floor": 1,
    "rooms": 1,
    "building_type": "G?ishtli",
    "area": 37,
}]
example_pred = float(predict_price(example_input)[0])
print(f"Predicted market price: {example_pred:.2f}")


### Quick reload for future sessions
If the model file already exists and you only need inference, you can skip training cells and run the next cell plus the prediction helper.


In [ ]:
# 6. Lightweight reload (use in future sessions)
model = joblib.load(MODEL_PATH)
